처음 한 번 실행

mkdir -p models

python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='unsloth/Qwen3-32B-GGUF', filename='Qwen3-32B-UD-Q6_K_XL.gguf', local_dir='./models', local_dir_use_symlinks=False)"

매 번 실행

export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH

./llama.cpp/build/bin/llama-server \
  -m ./models/Qwen3-32B-UD-Q6_K_XL.gguf \
  -c 13600 \
  -np 1 \
  -cb \
  -fa on \
  --port 8000 \
  --host 0.0.0.0

In [ ]:
import asyncio
import ast
import pandas as pd
from openai import AsyncOpenAI

model_name = "Qwen3-32B_RAG"
df_test = pd.read_csv('test_with_RAG.csv')
df_output = pd.read_csv('output.csv')

client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="sk-no-key-required",
    timeout=1200.0
)

In [ ]:
system_msg_0 = """당신은 객관식 문제를 해결하는 학생입니다. 먼저 제시문과 질문으로 이루어진 문제가 주어집니다. 그 뒤에는 문제를 해결하는 데에 참고할 수 있는 자료들이 주어집니다. 당신은 두 단계를 통해 문제를 해결해야 합니다.
첫 번째 단계에서는 주어진 참고자료들 중에서 문제의 내용과 관련이 깊고 해결과정에 꼭 필요한 자료들을 선별합니다. 주어지는 자료 중 대부분은 문제의 해결과정에 도움이 되지 않는 내용일 가능성이 높습니다. 따라서 매우 비판적인 시각으로 식별해야 하고, 문제풀이에 필요 없는 참고자료는 해결과정에서 과감하게 배제하십시오. 
두 번째 단계에서는 문제를 해결하기 위한 사고과정을 step-by-step으로 자세히 명시하면서 추론을 수행합니다. 이 때 정답에 이르는 과정을 논리적으로 작성해야 합니다.
풀이과정이 끝나면 맨 마지막 줄에는 아래의 형식으로 정답을 기재해주세요.
{"정답": "번호"}"""


system_msg_1 = """당신은 객관식 문제를 해결하는 학생입니다. 먼저 제시문과 질문으로 이루어진 문제가 주어집니다. 그 뒤에는 문제를 해결하는 데에 참고할 수 있는 자료들이 주어집니다. 
아래의 지시사항에 따라 매우 명료하고 근거 중심적인 reasoning을 작성하십시오.

지시사항:
1. 문제 분석:
  - 질문이 요구하는 핵심이 무엇인지 먼저 정의하십시오.
  - 필요한 배경지식이 있다면 어떤 지식인지 기술하고, 해당 지식이 현재 지문/문제에 제시되어 있는지 여부를 명시하십시오.
2. 참고자료 분석:
  - 주어진 참고자료들 중에서 문제 해결에 꼭 필요한 자료들을 선별하십시오. 문제풀이에 필요하지 않다고 판단되는 참고자료는 해결과정에서 과감하게 제외하십시오.
  - 각 자료가 문제 해결에 어떻게 기여하는지 설명하십시오.
3. 사고 과정 (CoT):
  - 각 선택지가 정답이거나 오답인 이유를 제시문에서 근거를 찾아 명확히 설명하십시오.
  - 단순히 정답만 맞히지 말고, 왜 나머지 선택지는 정답이 될 수 없는지(오답 소거)를 논리적으로 서술하십시오.
  - 논리적인 비약이 있거나 지식적인 근거가 부족한 지점을 솔직하게 서술하십시오.
4. 제출형식 준수:
  - 풀이 과정은 줄글로 작성하되, 불필요한 반복을 피하십시오.
  - 가장 마지막 줄에는 아래의 형식으로 답안을 제출하십시오.
{"정답": "번호"}"""


system_msg_2 = """You are a student solving multiple-choice questions. You will be provided with a problem consisting of a passage and a question, followed by reference materials to assist you. You must solve the problem through the following two steps:

Step 1: From the provided reference materials, select only those that are highly relevant and essential to solving the problem. It is highly likely that most of the materials provided will not be helpful. Therefore, you must identify them with a critical eye and decisively exclude any irrelevant references from the process.

Step 2: Perform reasoning by specifying your thought process in detail, step-by-step. You must logically describe the process of arriving at the correct answer.

After the reasoning process, provide the final answer on the very last line in the following format:
{"정답": "Number"}"""


system_msg_3 = """You are a student solving multiple-choice questions. You will be provided with a problem consisting of a passage and a question, followed by reference materials. Write a very clear, evidence-based reasoning according to the instructions below.

Instructions:
1. Problem Analysis:
  - Define the core requirement of the question.
  - Describe any necessary background knowledge and specify whether that knowledge is already provided in the passage or problem.
2. Reference Analysis:
  - Select only the essential materials needed to solve the problem. Decisively exclude any reference materials deemed unnecessary.
  - Explain how each selected material contributes to solving the problem.
3. Thought Process (Chain of Thought):
  - Clearly explain why each option is either correct or incorrect, using evidence found in the passage.
  - Do not just find the correct answer; provide a logical justification for why the other options are excluded (process of elimination).
  - Be honest about any points where there is a logical leap or a lack of supporting evidence.
4. Compliance with Submission Format:
  - Write the reasoning process in prose and avoid unnecessary repetition.
  - Submit the final answer on the very last line in the following format:
{"정답": "Number"}"""


list_system_msg = [system_msg_0, system_msg_1, system_msg_2, system_msg_3]
list_RAG = []
for index, row in df_test.iterrows():
    if pd.notna(row['RAG']):
        list_RAG.append(index)
print(list_RAG)

In [ ]:
async def get_inference(system_msg, user_content, seed):
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_content}
            ],
            max_tokens=7200,
            temperature=0.7,
            top_p=0.80,
            seed=seed,
            extra_body={"min_p": 0.0, "top_k": 20}
        )
        msg = response.choices[0].message
        return msg.reasoning_content + "\n" + msg.content if (hasattr(msg, 'reasoning_content') and msg.reasoning_content) else msg.content

    except Exception as e:
        print(f"Error: {e}")
        return f"Error: {e}"

async def process_row(index, row, seed_base):
    problem = ast.literal_eval(row['problems'])
    user_content = f"<제시문>\n{row['paragraph']}\n\n"
    if pd.notna(row['question_plus']):
        user_content += f"<보기>\n{row['question_plus']}\n\n"
    user_content += f"<질문>\n{problem['question']}\n"
    for k in range(len(problem['choices'])):
        user_content += f"{k+1}. {problem['choices'][k]}\n"
    user_content += "\n\n" + row['RAG']

    seed = (seed_base * len(df_test) + index)

    tasks = []
    for r in range(len(list_system_msg)):
        tasks.append(get_inference(list_system_msg[r], user_content, seed))
    
    results = await asyncio.gather(*tasks)
    
    return index, results

async def main():
    print(f"Start Inference with Model: {model_name}")
    
    for s in range(0, 2):
        print(f"=== Processing Loop s={s} ===")

        for i in list_RAG:
            print(f"Processing s={s}, i={i} (Parallel requests for {len(list_system_msg)} personas)...")
            
            idx, results = await process_row(i, df_test.loc[i], s)
            
            for r, output in enumerate(results):
                df_test.loc[idx, f'resp_{r}_{s}'] = output
            
            df_test.to_csv(f'TestSet_Inference_{model_name}.csv', index=False)
        
        df_test.to_csv(f'TestSet_Inference_{model_name}.csv', index=False)

In [ ]:
await main()